In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Setup

## install requirements

In [2]:
# DONE IIN MANAGE DEPENDENCIES

## Imports

In [3]:
import os
import gc
import json
import random

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from gensim.models import Word2Vec

import torch

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

import wandb

from kaggle_secrets import UserSecretsClient

from huggingface_hub import login, whoami

import string


## Seeding for Reproducibility

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Weights and Biases Setup

In [5]:
user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANB_API_KEY")


wandb.login(
    key=wandb_api_key
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: siddhartha12122002 (24f1002705-IITM) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## HuggingFace Setup (for Deploying)

In [6]:
hf_token = user_secrets.get_secret(
    "hf_token"
)

login(hf_token)

whoami()

{'type': 'user',
 'id': '69149e0c8c395b26b9d30cb6',
 'name': 'SiddharthaVarma12',
 'fullname': 'Siddhartha Varma',
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1782864000,
 'isPro': False,
 'avatarUrl': '/avatars/54e0ed6c83bbd2e679fbe09fd23af9c3.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'NPPE2',
   'role': 'fineGrained',
   'createdAt': '2025-12-15T10:22:57.136Z',
   'fineGrained': {'canReadGatedRepos': True,
    'global': ['discussion.write', 'post.write'],
    'scoped': [{'entity': {'_id': '69149e0c8c395b26b9d30cb6',
       'type': 'user',
       'name': 'SiddharthaVarma12'},
      'permissions': ['repo.content.read',
       'repo.write',
       'inference.serverless.write',
       'inference.endpoints.infer.write',
       'inference.endpoints.write',
       'user.webhooks.read',
       'user.webhooks.write',
       'collection.read',
       'collection.write',
       'discussion.write',
       'user.billing.read',
       'job

## MAP@3 function

In [7]:
# takes two inputs, one is a list of all correct answers, the other a list of corresponding predictions
def map_at_3(y_true, y_pred):
    score = 0.0

    if len(y_true) != len(y_pred):
        raise ValueError("Length mismatch")

    for i in range(len(y_true)):
        pred = y_pred[i]
        true = y_true[i]
        if true in pred:
            for j in range(len(pred)):
                if pred[j] == true:
                    score += 1/(j+1)
                    break
                
    return score/len(y_true)

## Data cleaning

In [8]:
import re

def clean_text(text):

    text = str(text).lower()

    text = re.sub(
        r"[^a-z0-9 ]",
        " ",
        text
    )

    return text

# Exploratory Data Analysis

In [9]:
df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
df

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A
...,...,...,...,...,...,...,...,...
1995,1996,What is the piezoelectric strain coefficient f...,d = 1.9·10‑12 m/V,d = 3.1·10‑12 m/V,d = 4.2·10‑12 m/V,d = 2.5·10‑12 m/V,d = 5.8·10‑12 m/V,B
1996,1997,Identify the correct statement: What is the sy...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,A device used to demonstrate a neuro-inspired ...,E
1997,1998,Determine the correct option: What does Earnsh...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges can be maintaine...,A collection of point charges cannot be mainta...,A collection of point charges can be maintaine...,D
1998,1999,Identify the correct statement: What is the re...,The atmosphere is a mechanism that is only inf...,"The atmosphere possesses both chaos and order,...",The atmosphere is a structure that is only inf...,The atmosphere is a completely chaotic mechani...,The atmosphere is a completely ordered structu...,B


In [10]:
df_test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
df_test

,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
...,...,...,...,...,...,...,...
495,496,What are the constituents of cold dark matter?,"They are unknown, but possibilities include la...",They are known to be black holes and Preon stars.,They are only MACHOs.,They are clusters of brown dwarfs.,They are new particles such as RAMBOs.
496,497,Pick the best possible answer: What is a plane...,A framework of planets that are all located in...,A mechanism of planets that are all the same s...,Any set of gravitationally bound non-stellar o...,A mechanism of planets that are all located in...,A structure of planets that are all made of gas.
497,498,Pick the best possible answer: What is magneti...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...,Magnetic susceptibility is a measure of how mu...
498,499,Determine the correct option: What is the evid...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The Milky Way galaxy has a supermassive black ...,The star S2 follows an elliptical orbit with a...


In [11]:
df['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [12]:
df.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

In [13]:
# Calculating the unique words in cleaned prompts
cleaned = (
    df["prompt"]
    .astype(str)
    .str.lower()
    .str.translate(str.maketrans('', '', string.punctuation))
)

# Split by whitespace and collect all unique words
vocab = set()

for text in cleaned:
    vocab.update(text.split())

print("Vocabulary size:", len(vocab))

Vocabulary size: 859


# Milestone-1: Tfidf and Word2Vec

## tfidf

In [14]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["answer"]
)

In [15]:
train_text = (
    train_df["prompt"].fillna("") + " " +
    train_df["A"].fillna("") + " " +
    train_df["B"].fillna("") + " " +
    train_df["C"].fillna("") + " " +
    train_df["D"].fillna("") + " " +
    train_df["E"].fillna("")
)

vectorizer1 = TfidfVectorizer(
    stop_words="english"
)

vectorizer1.fit(train_text)

TfidfVectorizer(stop_words='english')

In [16]:
full_text = (
    df["prompt"].fillna("") + " " +
    df["A"].fillna("") + " " +
    df["B"].fillna("") + " " +
    df["C"].fillna("") + " " +
    df["D"].fillna("") + " " +
    df["E"].fillna("")
)

vectorizer2 = TfidfVectorizer(
    stop_words="english"
)

vectorizer2.fit(full_text)

TfidfVectorizer(stop_words='english')

In [17]:
def predict_score(vectorizer, df):
    predictions = []
    
    for _, row in df.iterrows():
    
        prompt_vec = vectorizer.transform(
            [row["prompt"]]
        )
    
        similarities = {}
    
        for option in ["A","B","C","D","E"]:
    
            option_vec = vectorizer.transform(
                [row[option]]
            )
    
            similarities[option] = cosine_similarity(
                prompt_vec,
                option_vec
            )[0,0]
    
        ranked = sorted(
            similarities,
            key=similarities.get,
            reverse=True
        )
    
        predictions.append(
            ranked[:3]
        )
    
    score = map_at_3(
        df["answer"].tolist(),
        predictions
    )
    
    print("Validation MAP@3:", score)

    return score

def tfidf_predict(vectorizer, df):
    predictions = []
    
    for _, row in df.iterrows():
    
        prompt_vec = vectorizer.transform(
            [row["prompt"]]
        )
    
        similarities = {}
    
        for option in ["A","B","C","D","E"]:
    
            option_vec = vectorizer.transform(
                [row[option]]
            )
    
            similarities[option] = cosine_similarity(
                prompt_vec,
                option_vec
            )[0,0]
    
        ranked = sorted(
            similarities,
            key=similarities.get,
            reverse=True
        )
    
        predictions.append(
            ranked[:3]
        )

    return predictions

Testing and Logging val_text tfidf (split data)

In [18]:
'''run = wandb.init(
    entity="24f1002705-IITM",
    project="24f1002705-DLGenAI-t12026",
    config={
        "model": "tfidf-split",
        "stop_words": "english"
    },
)

score = predict_score(vectorizer1, val_df)

run.log({
    "map3": score
})

run.finish()'''

'run = wandb.init(\n    entity="24f1002705-IITM",\n    project="24f1002705-DLGenAI-t12026",\n    config={\n        "model": "tfidf-split",\n        "stop_words": "english"\n    },\n)\n\nscore = predict_score(vectorizer1, val_df)\n\nrun.log({\n    "map3": score\n})\n\nrun.finish()'

Testing and Logging fulltext tfidf

In [19]:
'''run = wandb.init(
    entity="24f1002705-IITM",
    project="24f1002705-DLGenAI-t12026",
    config={
        "model": "tfidf-full",
        "stop_words": "english"
    },
)

score = predict_score(vectorizer2, df)

run.log({
    "map3": score
})

run.finish()'''

'run = wandb.init(\n    entity="24f1002705-IITM",\n    project="24f1002705-DLGenAI-t12026",\n    config={\n        "model": "tfidf-full",\n        "stop_words": "english"\n    },\n)\n\nscore = predict_score(vectorizer2, df)\n\nrun.log({\n    "map3": score\n})\n\nrun.finish()'

In [20]:
prediction1 = tfidf_predict(vectorizer1, df_test)
y_pred1 = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [" ".join(pred) for pred in prediction1]
})
y_pred1.to_csv('submissionTfidfSplit.csv', index=False)

In [21]:
prediction2 = tfidf_predict(vectorizer2, df_test)
y_pred1 = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [" ".join(pred) for pred in prediction2]
})
y_pred1.to_csv('submissionTfidfFull.csv', index=False)

### Submission Analysis
* TfidfSplit - 0.32252
* TfidfFull - 0.32294
Worse than baseline 'A B C' for all options, not for consideration

## Word2Vec

In [22]:
# building the corpus
corpus = []

for _, row in train_df.iterrows():
    corpus.append(
        str(clean_text(row['prompt'])).lower().split()
    )

    for option in ["A", "B", "C", "D", "E"]:
        corpus.append(
            str(row[option]).lower().split()
        )

In [23]:
# Train word2vec
w2v_model = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    seed=42
)

In [24]:
def sentence_embedding(text, model):

    words = clean_text(str(text)).lower().split()

    vectors = []

    for word in words:

        if word in model.wv:

            vectors.append(
                model.wv[word]
            )

    if len(vectors) == 0:

        return np.zeros(
            model.vector_size
        )

    return np.mean(
        vectors,
        axis=0
    )

### Validation score

In [25]:
predictions = []

for _, row in val_df.iterrows():

    prompt_emb = sentence_embedding(
        clean_text(row["prompt"]),
        w2v_model
    )

    similarities = {}

    for option in ["A","B","C","D","E"]:

        option_emb = sentence_embedding(
            row[option],
            w2v_model
        )

        sim = cosine_similarity(
            prompt_emb.reshape(1,-1),
            option_emb.reshape(1,-1)
        )[0,0]

        similarities[option] = sim

    ranked = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    predictions.append(
        ranked[:3]
    )

In [26]:
score = map_at_3(val_df['answer'].tolist(), predictions)
score

0.36

In [27]:
'''wandb.init(
    entity="24f1002705-IITM",
    project="24f1002705-DLGenAI-t12026",
    name="word2vec-baseline"
)

wandb.log({
    "map3": score,
    "vector_size": 100,
    "window": 5,
    "min_count": 1
})

wandb.finish()'''

'wandb.init(\n    entity="24f1002705-IITM",\n    project="24f1002705-DLGenAI-t12026",\n    name="word2vec-baseline"\n)\n\nwandb.log({\n    "map3": score,\n    "vector_size": 100,\n    "window": 5,\n    "min_count": 1\n})\n\nwandb.finish()'

### Submission

In [28]:
test_predictions = []

for _, row in df_test.iterrows():

    prompt_emb = sentence_embedding(
        clean_text(row["prompt"]),
        w2v_model
    )

    similarities = {}

    for option in ["A","B","C","D","E"]:

        option_emb = sentence_embedding(
            row[option],
            w2v_model
        )

        sim = cosine_similarity(
            prompt_emb.reshape(1,-1),
            option_emb.reshape(1,-1)
        )[0,0]

        similarities[option] = sim

    ranked = sorted(
        similarities,
        key=similarities.get,
        reverse=True
    )

    test_predictions.append(
        ranked[:3]
    )

In [29]:
submission = pd.DataFrame({
    'ID': df_test['id'],
    'Prediction': [" ".join(pred) for pred in test_predictions]
})

submission.to_csv(
    "word2vec_submission.csv",
    index=False
)

In [30]:
submission

,ID,Prediction
0,1,C A D
1,2,B D E
2,3,A D C
3,4,A C E
4,5,B D E
...,...,...
495,496,D E B
496,497,B D A
497,498,D E A
498,499,D A B
